In [1]:
from generate_utils import load_GraphModel, load_BiLSTMModel, load_TokenBiLSTMModel, load_LoRASEModel, load_AdapterModel
import torch
import numpy as np
import pickle
from GridMLM_tokenizers import CSGridMLMTokenizer
import os
from eval_utils import get_vecser_for_file, vecser_similarity_matrix
from dotenv import load_dotenv
from tqdm import tqdm
from eval_utils import vecser_similarity_evidence_for_files

from langchain_ollama import ChatOllama
from langchain.tools import tool

# Load environment variables from .env file
load_dotenv()

/home/maximos/miniconda3/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# Initialize the ChatOllama model with the specified model name
model_name = 'qwen2.5-coder:7b'

# and initialize the ChatOllama instance
chat_model = ChatOllama(
    model=model_name,
    validate_model_on_init=True,
    temperature=0.7
)

In [3]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

def absoluteFilePaths(directory):
    file_names = []
    file_paths = []
    for dirpath,_,filenames in os.walk(directory):
        for f in filenames:
            if f.endswith( ('.mid', '.midi', '.mxl', '.xml', '.musicxml') ):
                file_names.append(f)
                file_paths.append(os.path.abspath(os.path.join(dirpath, f)))
    return file_names, file_paths
# end absoluteFilePaths

hook_file_names, hook_file_paths = absoluteFilePaths(os.getenv('VAL_HOOK'))
gjt_file_names, gjt_file_paths = absoluteFilePaths(os.getenv('VAL_GJT'))

device_name = 'cuda:2'
device = torch.device(device_name)

guide_arch = 'LoRA'
contra = True

adapter_model_path = f'saved_models/{guide_arch}/adapter/adapter_model_' + contra*'contra_' + 'jnhw.pt'
graph_adapter_model_path = f'saved_models/{guide_arch}/adapter/graph_model_' + contra*'contra_' + 'jnhw.pt'
token_adapter_model_path = f'saved_models/{guide_arch}/adapter/bilstm_model_' + contra*'contra_' + 'jnhw.pt'

token_adapter_model = load_TokenBiLSTMModel(token_adapter_model_path, tokenizer, device)
graph_adapter_model = load_GraphModel(graph_adapter_model_path, device)
adapter_model = load_AdapterModel(adapter_model_path, device)

token_adapter_model.eval()
graph_adapter_model.eval()
adapter_model.eval()

GuidanceAdapter(
  (proj): Linear(in_features=1024, out_features=512, bias=True)
)

In [4]:
f1 = gjt_file_paths[0]
f2 = gjt_file_paths[1]
print(f1)
print(f2)

bars_string, graph_res, token_res, adapter_res = vecser_similarity_evidence_for_files(
    f1,
    f2,
    tokenizer,
    graph_model=graph_adapter_model,
    bilstm_model=None,
    token_model=token_adapter_model,
    adapter_model=adapter_model,
    topk=10
)

/media/maindisk/data/mel_harm_CA_all/gjt_CA_test/Mean_To_Me.mxl
/media/maindisk/data/mel_harm_CA_all/gjt_CA_test/My_One_And_Only_Love.mxl


In [5]:
print(bars_string)

Piece 1:
bar 0: G:maj6 G#:dim7 
bar 1: A:min7 D:7 
bar 2: G:maj7 D:min7 G:7 
bar 3: C:maj7 F:9 
bar 4: G:maj7 E:7 
bar 5: A:min7 D:7 
bar 6: G:maj6 E:min7 
bar 7: A:7 D:7 
bar 8: G:maj6 C:maj6 
bar 9: G:maj6 D:min7 G:7 
bar 10: C:maj6 
bar 11: D:9 G:7 
bar 12: C:maj6 
bar 13: F:7 E:7 
bar 14: A:min7 
bar 15: F:7 E:7 

Piece 2:
bar 0: F:maj7 D:min7 
bar 1: G:min7 C:7 C#:dim 
bar 2: D:min7 A#:maj7 
bar 3: A:min7 D:7 
bar 4: G:min7 C:7 C#:dim 
bar 5: D:min7 G:7 
bar 6: G:min7 C:7 
bar 7: A:min7 D:7 G:min7 C:7 
bar 8: F:maj6 B:hdim7 E:7 
bar 9: A:min7 
bar 10: B:hdim7 E:7(b9) 
bar 11: A:min7 
bar 12: B:hdim7 E:7(b9) 
bar 13: A:min A:minmaj7 
bar 14: A:min7 D:7 
bar 15: G:min7 D:7 



In [6]:
print(graph_res)

Graph model evidence:
piece 1, bar 12: ['C:maj6'] | piece 2, bar 11: ['A:min7'] | 0.9996343851089478
piece 1, bar 2: ['G:maj7', 'D:min7', 'G:7'] | piece 2, bar 5: ['D:min7', 'G:7'] | 0.7704873085021973
piece 1, bar 7: ['A:7', 'D:7'] | piece 2, bar 3: ['A:min7', 'D:7'] | 0.7364344596862793
piece 1, bar 8: ['G:maj6', 'C:maj6'] | piece 2, bar 14: ['A:min7', 'D:7'] | 0.7003163695335388
piece 1, bar 11: ['D:9', 'G:7'] | piece 2, bar 5: ['D:min7', 'G:7'] | 0.6958389282226562
piece 1, bar 0: ['G:maj6', 'G#:dim7'] | piece 2, bar 10: ['B:hdim7', 'E:7(b9)'] | 0.6921545267105103
piece 1, bar 1: ['A:min7', 'D:7'] | piece 2, bar 5: ['D:min7', 'G:7'] | 0.6828776597976685
piece 1, bar 7: ['A:7', 'D:7'] | piece 2, bar 15: ['G:min7', 'D:7'] | 0.6638665199279785
piece 1, bar 5: ['A:min7', 'D:7'] | piece 2, bar 15: ['G:min7', 'D:7'] | 0.6544992923736572
piece 1, bar 9: ['G:maj6', 'D:min7', 'G:7'] | piece 2, bar 5: ['D:min7', 'G:7'] | 0.6302633285522461



In [7]:
print(token_res)

Token model evidence:
piece 1, bar 7: ['A:7', 'D:7'] | piece 2, bar 3: ['A:min7', 'D:7'] | 0.7414132952690125
piece 1, bar 1: ['A:min7', 'D:7'] | piece 2, bar 15: ['G:min7', 'D:7'] | 0.7253557443618774
piece 1, bar 7: ['A:7', 'D:7'] | piece 2, bar 15: ['G:min7', 'D:7'] | 0.6307624578475952
piece 1, bar 6: ['G:maj6', 'E:min7'] | piece 2, bar 8: ['F:maj6', 'B:hdim7', 'E:7'] | 0.5227781534194946
piece 1, bar 14: ['A:min7'] | piece 2, bar 4: ['G:min7', 'C:7', 'C#:dim'] | 0.5074173808097839
piece 1, bar 8: ['G:maj6', 'C:maj6'] | piece 2, bar 13: ['A:min', 'A:minmaj7'] | 0.5039945840835571
piece 1, bar 12: ['C:maj6'] | piece 2, bar 13: ['A:min', 'A:minmaj7'] | 0.5002508163452148
piece 1, bar 2: ['G:maj7', 'D:min7', 'G:7'] | piece 2, bar 5: ['D:min7', 'G:7'] | 0.494163453578949
piece 1, bar 14: ['A:min7'] | piece 2, bar 3: ['A:min7', 'D:7'] | 0.4927017092704773
piece 1, bar 5: ['A:min7', 'D:7'] | piece 2, bar 11: ['A:min7'] | 0.4927017092704773



In [8]:
print(adapter_res)

Adapter model evidence:
piece 1, bar 10: ['C:maj6'] | piece 2, bar 11: ['A:min7'] | 0.939810037612915
piece 1, bar 7: ['A:7', 'D:7'] | piece 2, bar 3: ['A:min7', 'D:7'] | 0.8129514455795288
piece 1, bar 1: ['A:min7', 'D:7'] | piece 2, bar 15: ['G:min7', 'D:7'] | 0.7870017886161804
piece 1, bar 7: ['A:7', 'D:7'] | piece 2, bar 15: ['G:min7', 'D:7'] | 0.7619063258171082
piece 1, bar 8: ['G:maj6', 'C:maj6'] | piece 2, bar 13: ['A:min', 'A:minmaj7'] | 0.7256702780723572
piece 1, bar 1: ['A:min7', 'D:7'] | piece 2, bar 0: ['F:maj7', 'D:min7'] | 0.7003436088562012
piece 1, bar 5: ['A:min7', 'D:7'] | piece 2, bar 5: ['D:min7', 'G:7'] | 0.6731694936752319
piece 1, bar 2: ['G:maj7', 'D:min7', 'G:7'] | piece 2, bar 5: ['D:min7', 'G:7'] | 0.6675118207931519
piece 1, bar 3: ['C:maj7', 'F:9'] | piece 2, bar 0: ['F:maj7', 'D:min7'] | 0.6441857814788818
piece 1, bar 11: ['D:9', 'G:7'] | piece 2, bar 5: ['D:min7', 'G:7'] | 0.6413013935089111

